<a href="https://colab.research.google.com/github/Minimire/README.md/blob/main/Copy_of_Implementation_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

# ============================================================
# CELL 1 — Install dependencies + load dataset
# ============================================================
!pip -q install xgboost shap optuna openpyxl joblib

import os, warnings, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# --- 3-tier dataset loader ---
DATA_XLSX = "Crop_recommendation.xlsx"
DATA_CSV  = "Crop_recommendation.csv"
URL       = ("https://raw.githubusercontent.com/Gladiator07/Harvestify/"
             "master/Data-processed/crop_recommendation.csv")

if os.path.exists(DATA_XLSX):
    df = pd.read_excel(DATA_XLSX); print(f"✔ Loaded {DATA_XLSX}")
elif os.path.exists(DATA_CSV):
    df = pd.read_csv(DATA_CSV);   print(f"✔ Loaded {DATA_CSV}")
else:
    df = pd.read_csv(URL);        print(f"✔ Loaded from fallback URL (same Kaggle data)")

print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head()

In [ ]:

# ============================================================
# CELL 2 — Quality checks + EDA
# ============================================================
FEATURES = ["N", "P", "K", "temperature", "humidity", "ph", "rainfall"]

# Quality
print("Missing values total:", int(df.isnull().sum().sum()))
print("Duplicate rows       :", int(df.duplicated().sum()))
print("Class count          :", df["label"].nunique())
print("Samples per class    :", df["label"].value_counts().unique())
print("Soil pH mean         :", round(df["ph"].mean(), 2))
print("Rainfall range (mm)  :", round(df["rainfall"].min(), 1), "→", round(df["rainfall"].max(), 1))

# Plot 1: distributions
fig, axes = plt.subplots(2, 4, figsize=(18, 7)); axes = axes.flatten()
for i, col in enumerate(FEATURES):
    sns.histplot(df[col], kde=True, ax=axes[i], color="seagreen")
    axes[i].set_title(col)
axes[-1].axis("off"); plt.tight_layout(); plt.show()

# Plot 2: correlation
plt.figure(figsize=(8, 6))
sns.heatmap(df[FEATURES].corr(), annot=True, fmt=".2f", cmap="YlGnBu", square=True)
plt.title("Feature Correlations"); plt.show()

# Plot 3: rainfall per crop (Rice threshold insight)
plt.figure(figsize=(15, 5))
sns.boxplot(data=df, x="label", y="rainfall", palette="Set3")
plt.xticks(rotation=90); plt.title("Rainfall per Crop"); plt.tight_layout(); plt.show()

print("\nTop-5 high-potassium crops:")
print(df.groupby("label")["K"].mean().sort_values(ascending=False).head(5).round(1))

In [ ]:

# ============================================================
# CELL 3 — Preprocessing
# ============================================================
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

X  = df[FEATURES].copy()
y_raw = df["label"].copy()

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_raw)

print("Encoded classes (22):")
for i, c in enumerate(label_encoder.classes_):
    print(f"  {i:2d} → {c}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
print(f"\nTrain: {X_train.shape}  |  Test: {X_test.shape}")

# Scaler fitted on TRAIN only (no leakage)
scaler = StandardScaler().fit(X_train)

# Save preprocessing artifacts
joblib.dump({"features": FEATURES, "label_encoder": label_encoder,
             "scaler": scaler}, "preprocessing.pkl")
print("✔ Saved preprocessing.pkl")

In [ ]:

# ============================================================
# CELL 3b — Seed-robustness audit (RF + XGB)
# ============================================================
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier
import numpy as np

SEEDS = [0, 42, 123]

rf_accs, xgb_accs = [], []

for seed in SEEDS:
    Xtr, Xte, ytr, yte = train_test_split(
        X, y, test_size=0.20, random_state=seed, stratify=y)

    # RF — model seed fixed, split seed varies
    rf = RandomForestClassifier(
        n_estimators=500, random_state=42, n_jobs=-1).fit(Xtr, ytr)
    rf_acc = accuracy_score(yte, rf.predict(Xte))
    rf_accs.append(rf_acc)

    # XGB — same protocol
    xgb = XGBClassifier(
        n_estimators=200, learning_rate=0.1, max_depth=5,
        objective="multi:softprob", num_class=22,
        eval_metric="mlogloss", tree_method="hist",
        random_state=42, n_jobs=-1).fit(Xtr, ytr)
    xgb_acc = accuracy_score(yte, xgb.predict(Xte))
    xgb_accs.append(xgb_acc)

    print(f"Seed {seed:3d} | RF = {rf_acc:.4f} ({rf_acc*100:.2f}%) "
          f"| XGB = {xgb_acc:.4f} ({xgb_acc*100:.2f}%)")

print("\n" + "="*60)
print(f"Random Forest : {np.mean(rf_accs):.4f} ± {np.std(rf_accs):.4f} "
      f"({np.mean(rf_accs)*100:.2f}% ± {np.std(rf_accs)*100:.2f}%)")
print(f"XGBoost       : {np.mean(xgb_accs):.4f} ± {np.std(xgb_accs):.4f} "
      f"({np.mean(xgb_accs)*100:.2f}% ± {np.std(xgb_accs)*100:.2f}%)")
print("="*60)
print(f"Range (RF): {min(rf_accs)*100:.2f}% → {max(rf_accs)*100:.2f}%")

In [ ]:

# ============================================================
# CELL 4 — Train & compare all models
# ============================================================
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score)

models = {
    "Dummy (control)": DummyClassifier(strategy="most_frequent"),
    "Logistic Regression": Pipeline([("sc", StandardScaler()),
                                     ("clf", LogisticRegression(max_iter=2000,
                                                                random_state=RANDOM_STATE))]),
    "KNN (k=5)": Pipeline([("sc", StandardScaler()),
                           ("clf", KNeighborsClassifier(n_neighbors=5))]),
    "Random Forest": RandomForestClassifier(n_estimators=500,
                                            random_state=RANDOM_STATE,
                                            n_jobs=-1),
    "XGBoost": XGBClassifier(n_estimators=200, learning_rate=0.1, max_depth=5,
                             objective="multi:softprob", num_class=22,
                             eval_metric="mlogloss", tree_method="hist",
                             random_state=RANDOM_STATE),
}

rows, trained = [], {}
for name, m in models.items():
    m.fit(X_train, y_train)
    p = m.predict(X_test)
    trained[name] = m
    rows.append({"Model": name,
                 "Accuracy":  accuracy_score(y_test, p),
                 "Precision": precision_score(y_test, p, average="macro", zero_division=0),
                 "Recall":    recall_score(y_test, p, average="macro", zero_division=0),
                 "F1":        f1_score(y_test, p, average="macro", zero_division=0)})

results = pd.DataFrame(rows).sort_values("Accuracy", ascending=False).reset_index(drop=True)
display(results.style.format({"Accuracy":"{:.2%}","Precision":"{:.2%}",
                              "Recall":"{:.2%}","F1":"{:.2%}"}))

In [ ]:

# ============================================================
# CELL 5 — Why your number differs from the README (10-seed audit)
# ============================================================
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import numpy as np

rf_accs = []
for seed in range(10):
    Xtr, Xte, ytr, yte = train_test_split(
        X, y, test_size=0.20, stratify=y, random_state=seed)
    m = RandomForestClassifier(n_estimators=500, random_state=RANDOM_STATE, n_jobs=-1)
    m.fit(Xtr, ytr)
    rf_accs.append(accuracy_score(yte, m.predict(Xte)))

print("Random Forest across 10 random splits:")
print("  Per-seed:", [f"{a:.2%}" for a in rf_accs])
print(f"  Mean ± std: {np.mean(rf_accs):.4f} ± {np.std(rf_accs):.4f}")
print(f"  Range:      {min(rf_accs):.2%} → {max(rf_accs):.2%}")
print("\n→ README reports 99.55% (a single split). This is a legitimate result, "
      "but note the ±std above is larger than the gap between RF and XGBoost.")

In [ ]:

# ============================================================
# CELL 6 — Tuned Random Forest (GridSearchCV)
# ============================================================
from sklearn.model_selection import GridSearchCV

param_grid = {
    "n_estimators": [200, 500, 800],
    "max_depth":    [None, 15, 25],
    "max_features": ["sqrt", "log2"],
}

gs = GridSearchCV(
    RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
    param_grid, cv=5, scoring="accuracy", n_jobs=-1, verbose=1
)
gs.fit(X_train, y_train)

rf_best = gs.best_estimator_
rf_test_acc = accuracy_score(y_test, rf_best.predict(X_test))

print("\nBest params:", gs.best_params_)
print(f"Best CV acc : {gs.best_score_:.4f}")
print(f"Test acc    : {rf_test_acc:.4f}  ({rf_test_acc:.2%})")

In [ ]:

# ============================================================
# CELL 7 — Tuned XGBoost (Optuna, 20 trials)
# ============================================================
import optuna
from sklearn.model_selection import cross_val_score
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    params = dict(
        n_estimators     = trial.suggest_int("n_estimators", 50, 300),
        max_depth        = trial.suggest_int("max_depth", 3, 10),
        learning_rate    = trial.suggest_float("learning_rate", 0.05, 0.3, log=True),
        subsample        = trial.suggest_float("subsample", 0.7, 1.0),
        colsample_bytree = trial.suggest_float("colsample_bytree", 0.7, 1.0),
        objective="multi:softprob", num_class=22, eval_metric="mlogloss",
        tree_method="hist", random_state=RANDOM_STATE, n_jobs=-1,
    )
    return cross_val_score(XGBClassifier(**params),
                           X_train, y_train, cv=3,
                           scoring="accuracy", n_jobs=-1).mean()

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=20, show_progress_bar=True)

best_xgb = XGBClassifier(
    **study.best_params,
    objective="multi:softprob", num_class=22, eval_metric="mlogloss",
    tree_method="hist", random_state=RANDOM_STATE, n_jobs=-1
).fit(X_train, y_train)

print(f"\nBest CV acc : {study.best_value:.4f}")
print(f"Best params : {study.best_params}")
print(f"Test acc    : {accuracy_score(y_test, best_xgb.predict(X_test)):.4f}")

In [ ]:

# ============================================================
# CELL 8 — Final model, classification report, SHAP
# ============================================================
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
import shap

# Pick the better of the two tuned models
final_model = rf_best if rf_test_acc >= accuracy_score(
    y_test, best_xgb.predict(X_test)) else best_xgb
final_name  = type(final_model).__name__
final_acc   = accuracy_score(y_test, final_model.predict(X_test))

print(f"Final model: {final_name}  |  Test accuracy: {final_acc:.4f} ({final_acc:.2%})\n")
print(classification_report(y_test, final_model.predict(X_test),
                            target_names=label_encoder.classes_, digits=4))

# Confusion matrix
fig, ax = plt.subplots(figsize=(13, 11))
ConfusionMatrixDisplay.from_predictions(
    y_test, final_model.predict(X_test),
    display_labels=label_encoder.classes_,
    xticks_rotation=90, cmap="Greens", ax=ax, colorbar=False)
plt.title(f"{final_name} — Confusion Matrix"); plt.tight_layout(); plt.show()

# SHAP (API-safe across shap versions)
explainer = shap.TreeExplainer(final_model)
sv = explainer.shap_values(X_test)
if isinstance(sv, list):
    mean_abs = np.mean([np.abs(s).mean(axis=0) for s in sv], axis=0)
else:
    arr = np.asarray(sv)
    mean_abs = np.abs(arr).mean(axis=(0, 2)) if arr.ndim == 3 else np.abs(arr).mean(axis=0)

importance = pd.DataFrame({"Feature": FEATURES, "Mean |SHAP|": mean_abs}) \
               .sort_values("Mean |SHAP|", ascending=False).reset_index(drop=True)
print("\nSHAP feature importance ranking:")
display(importance)

plt.figure(figsize=(8, 4))
sns.barplot(data=importance, x="Mean |SHAP|", y="Feature", palette="viridis")
plt.title("SHAP Feature Importance"); plt.tight_layout(); plt.show()

In [ ]:

# ============================================================
# CELL 9 — Functional test cases
# ============================================================
def predict_crop(N, P, K, temperature, humidity, ph, rainfall, model=final_model):
    row = pd.DataFrame([{"N": N, "P": P, "K": K,
                         "temperature": temperature, "humidity": humidity,
                         "ph": ph, "rainfall": rainfall}])[FEATURES]
    proba    = model.predict_proba(row)[0]
    idx      = int(proba.argmax())
    return label_encoder.inverse_transform([idx])[0], float(proba[idx])

test_cases = [
    dict(name="TC1: README 'normal' values",
         inputs=dict(N=90, P=42, K=43, temperature=20.8,
                     humidity=82.1, ph=6.5, rainfall=202.9),
         readme_expected="Maize"),
    dict(name="TC2: Drought scenario",
         inputs=dict(N=20, P=40, K=20, temperature=48.0,
                     humidity=45.0, ph=7.0, rainfall=5.0),
         readme_expected="Moth Beans"),
    dict(name="TC3: Rice-like conditions (high rain, high humidity)",
         inputs=dict(N=90, P=40, K=40, temperature=23.0,
                     humidity=82.0, ph=6.5, rainfall=230.0),
         readme_expected="Rice"),
    dict(name="TC4: Wheat/cereal dry conditions",
         inputs=dict(N=80, P=48, K=20, temperature=22.0,
                     humidity=63.0, ph=6.8, rainfall=75.0),
         readme_expected="Maize"),
]

out = []
for tc in test_cases:
    crop, conf = predict_crop(**tc["inputs"])
    match = crop.lower() == tc["readme_expected"].lower()
    out.append({
        "Test Case": tc["name"],
        "Predicted": crop.title(),
        "Confidence": f"{conf:.1%}",
        "README expects": tc["readme_expected"],
        "Status": "✅ MATCH" if match else "⚠️ DIFFERS",
    })

display(pd.DataFrame(out))

# NOTE about TC1:
# N=90, P=42, K=43, T=20.8, H=82.1, pH=6.5, R=202.9 is the *first row*
# of the Kaggle dataset and is labeled RICE, not Maize. Every RF trained on
# this data outputs "Rice" here. The README's claim of "Maize @ 98%" for
# these exact inputs appears to be a documentation error.

In [ ]:

# ============================================================
# CELL 10 — Save model artifact for Streamlit
# ============================================================
artifact = {
    "model":           final_model,
    "label_encoder":   label_encoder,
    "scaler":          scaler,
    "features":        FEATURES,
    "classes":         list(label_encoder.classes_),
    "test_accuracy":   float(final_acc),
    "sklearn_version": __import__("sklearn").__version__,
}
joblib.dump(artifact, "crop_recommendation_model.pkl")
print("✔ Saved crop_recommendation_model.pkl")
print(f"  Model : {final_name}")
print(f"  Acc   : {final_acc:.4f}")
print(f"  Classes: {len(artifact['classes'])}")

# Round-trip verification
loaded = joblib.load("crop_recommendation_model.pkl")
c, p = predict_crop(90, 42, 43, 20.8, 82.1, 6.5, 202.9, model=loaded["model"])
print(f"  Round-trip prediction: {c} ({p:.2%})")

In [ ]:

# ============================================================
# CELL 11 — Write Streamlit app + requirements
# ============================================================
%%writefile app.py
import streamlit as st
import pandas as pd
import joblib

st.set_page_config(page_title="Crop Recommendation", page_icon="🌱", layout="centered")

@st.cache_resource
def load_artifact():
    return joblib.load("crop_recommendation_model.pkl")

art      = load_artifact()
model    = art["model"]
le       = art["label_encoder"]
FEATURES = art["features"]

st.title("🌱 Intelligent Crop Recommendation System")
st.caption(f"FUSE AI-201 | Group 11 | {type(model).__name__} "
           f"({art['test_accuracy']:.2%} test accuracy)")

with st.form("inputs"):
    st.subheader("Soil measurements")
    c1, c2 = st.columns(2)
    with c1:
        N  = st.number_input("Nitrogen (N) — mg/kg",   0.0, 200.0, 90.0, 1.0)
        P  = st.number_input("Phosphorus (P) — mg/kg", 0.0, 200.0, 42.0, 1.0)
        K  = st.number_input("Potassium (K) — mg/kg",  0.0, 250.0, 43.0, 1.0)
        ph = st.slider("Soil pH",                      0.0, 14.0,   6.5, 0.1)
    with c2:
        temperature = st.number_input("Temperature (°C)", 0.0, 60.0, 20.8, 0.1)
        humidity    = st.slider("Humidity (%)",           0.0, 100.0, 82.0, 0.1)
        rainfall    = st.number_input("Rainfall (mm)",    0.0, 400.0, 202.9, 0.1)
    submitted = st.form_submit_button("🌾 Recommend Crop", use_container_width=True)

if submitted:
    row   = pd.DataFrame([{"N": N, "P": P, "K": K, "temperature": temperature,
                           "humidity": humidity, "ph": ph,
                           "rainfall": rainfall}])[FEATURES]
    proba = model.predict_proba(row)[0]
    idx   = int(proba.argmax())
    crop  = le.inverse_transform([idx])[0]
    conf  = float(proba[idx])

    st.success(f"### Recommended: **{crop.title()}**")
    st.metric("Confidence", f"{conf:.1%}")
    st.progress(min(conf, 1.0))

    top = proba.argsort()[::-1][:5]
    st.subheader("Top 5 alternatives")
    st.dataframe(pd.DataFrame({"Crop": le.inverse_transform(top),
                               "Probability": proba[top]})
                   .style.format({"Probability": "{:.2%}"}),
                 use_container_width=True, hide_index=True)

st.divider()
st.warning("**Responsible AI Notice** — Advisory only. "
           "Not a replacement for local soil tests or agronomist review.")

In [ ]:

%%writefile requirements.txt
streamlit>=1.30
pandas>=2.0
numpy>=1.24
scikit-learn>=1.3
xgboost>=2.0
shap>=0.44
optuna>=3.4
matplotlib>=3.7
seaborn>=0.13
openpyxl>=3.1
joblib>=1.3

In [ ]:

# ============================================================
# CELL 12 — Launch Streamlit via Cloudflare Tunnel
# ============================================================
!pip -q install streamlit

# Download cloudflared (single binary, no auth required)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
!cloudflared --version

import subprocess, time, re

# Start Streamlit in background
streamlit_proc = subprocess.Popen(
    ["streamlit", "run", "app.py",
     "--server.port", "8501",
     "--server.headless", "true",
     "--server.address", "0.0.0.0"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

time.sleep(8)  # let Streamlit boot

# Start tunnel and capture URL
print("Opening Cloudflare tunnel...\n")
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8501"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

url = None
for _ in range(300):
    line = tunnel_proc.stdout.readline()
    if not line:
        time.sleep(0.1); continue
    print(line, end="")
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m:
        url = m.group(0)
        print(f"\n\n{'='*60}\n✅  YOUR APP IS LIVE AT:\n   {url}\n{'='*60}\n")
        break

if not url:
    print("⚠️ Could not detect URL — look for the trycloudflare.com link above.")

Issue Before Now
Cell 15 deploy !pip install localtunnel ❌ cloudflared binary download ✅
RF untuned default n_estimators=300 GridSearchCV over 18 combos (Cell 6)
XGB untuned hardcoded params Optuna 20-trial search (Cell 7)
Accuracy gap unexplained Cell 5 audits 10 splits and shows ±std
Test Case 1 silently PASS/REVIEW Explicit ⚠️ DIFFERS flag + comment explaining why
SHAP API-version guard inline Still guarded; averaged importance bar chart
Model artifact dict with 7 keys Same, plus round-trip verification in Cell 10
Streamlit app st.cache_resource + form Same — those choices were correct